#### Environment Check

In [ ]:
# Find the lab folder whether this notebook is run from notebooks/ or retrieval-lab/.
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    LAB_DIR = PROJECT_DIR.parent
elif PROJECT_DIR.name == "retrieval-lab":
    LAB_DIR = PROJECT_DIR
else:
    LAB_DIR = PROJECT_DIR / "06-best-practices" / "retrieval-lab"

CODE_DIR = LAB_DIR / "code"

sys.path.append(str(CODE_DIR))

print("Python:", sys.executable)
print("Lab dir:", LAB_DIR)
print("Code dir:", CODE_DIR)

#### Load Search And RRF Helpers

In [ ]:
# Load search helpers plus the RRF reranking helper.
from elasticsearch import ConnectionError

from rrf import reciprocal_rank_fusion
from search import (
    create_es_client,
    create_embedding_model,
    keyword_search,
    vector_search,
    hybrid_search,
)

es_client = create_es_client()

try:
    es_info = es_client.info()
except ConnectionError as error:
    raise RuntimeError(
        "Elasticsearch is not running. From retrieval-lab, run: "
        "docker compose up -d"
    ) from error

embedding_model = create_embedding_model()

es_info

#### Check Elasticsearch Index

In [ ]:
# This index must exist before search can work. If it is missing, run code/ingest.py.
INDEX_NAME = "course-questions"

index_exists = es_client.indices.exists(index=INDEX_NAME)

print("Index exists:", index_exists)

if not index_exists:
    raise RuntimeError(
        "The Elasticsearch index is missing. From retrieval-lab, run: "
        "uv run python code/ingest.py"
    )

#### Query

In [ ]:
# Use the same query settings for every method so the comparison is fair.
query = "I just discovered the course. Can I still join?"
course = "data-engineering-zoomcamp"
num_results = 5

#### Run Keyword Search

In [ ]:
# First list: exact text matching with Elasticsearch keyword search.
keyword_results = keyword_search(
    query=query,
    course=course,
    num_results=num_results,
    es_client=es_client,
)

for i, doc in enumerate(keyword_results, start=1):
    print(i, doc["id"], doc["course"], "-", doc["question"])

#### Run Vector Search

In [ ]:
# Second list: semantic matching with the embedding vector field.
vector_results = vector_search(
    query=query,
    course=course,
    num_results=num_results,
    es_client=es_client,
    embedding_model=embedding_model,
)

for i, doc in enumerate(vector_results, start=1):
    print(i, doc["id"], doc["course"], "-", doc["question"])

#### Rerank With Reciprocal Rank Fusion

In [ ]:
# RRF merges keyword and vector rankings using rank positions, not raw scores.
rrf_results = reciprocal_rank_fusion(
    [keyword_results, vector_results],
    k=60,
    num_results=num_results,
)

for i, doc in enumerate(rrf_results, start=1):
    print(i, doc["id"], doc["course"], "-", doc["question"])

#### Compare With Elasticsearch Hybrid Search

In [ ]:
# Elasticsearch hybrid search is used here as a direct comparison to manual RRF.
hybrid_results = hybrid_search(
    query=query,
    course=course,
    num_results=num_results,
    es_client=es_client,
    embedding_model=embedding_model,
)

for i, doc in enumerate(hybrid_results, start=1):
    print(i, doc["id"], doc["course"], "-", doc["question"])

#### Compare All Results

In [ ]:
# Print all result lists in the same format for quick comparison.
def show_results(title, results):
    print(title)
    print("-" * len(title))

    for i, doc in enumerate(results, start=1):
        print(i, doc["id"], doc["course"], "-", doc["question"])

    print()


show_results("Keyword Search", keyword_results)
show_results("Vector Search", vector_results)
show_results("RRF Reranking", rrf_results)
show_results("Elasticsearch Hybrid Search", hybrid_results)

#### Why RRF Helps

In [ ]:
print("""
RRF combines ranked lists from different search methods.

A document gets a stronger final score when it appears high in more than one list.

This helps because:
- keyword search is good for exact words
- vector search is good for meaning
- RRF gives more weight to documents that both methods agree on
""")